# 🧠 CerebroSpheroid V5: High-Performance Colab Training & Cloud Sync
### Multi-Model Sync Hub (Small, Compact, Base) & Dataset Mixture Selector

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AjitKumar-git/cerebro_spheroid_v5/blob/main/colab/CerebroSpheroid_V5_Colab_Training.ipynb)

**Key Highlights:**
- **3 Model Tiers Supported:** `v5_small` (101M, 16 shells - salvaged V4 compatible), `v5_compact` (244M, 32 shells), and `v5_base` (617M, 48 shells).
- **Two-Way Multi-Model Cloud Sync:** Each model tier automatically synchronizes its own `{model}_latest.pt` and `{model}_best.pt` with `https://bioflo.in/api/cerebro/sync.php`.
- **Dedicated Dataset Mixtures:** Launch focused training on Foundation, Math, Code, Agentic Tools, or Golden Consolidation Replay from dedicated cells.
- **V4 Elastic Sliding Context Window:** Zero padding waste, complementary $(L_p, L_t)$ buckets, and preceding context feedback conditioning.

## 1. Verify GPU Hardware Acceleration & System Environment

In [ ]:
import torch, os, sys
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU:      {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:      {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Running on CPU! Go to Runtime -> Change runtime type -> Select T4 GPU or A100 for fast training.")

!nvidia-smi

## 2. Clone CerebroSpheroid V5 Repository & Install Dependencies

In [ ]:
%cd /content
if not os.path.exists("/content/cerebro_spheroid_v5"):
    !git clone https://github.com/AjitKumar-git/cerebro_spheroid_v5.git /content/cerebro_spheroid_v5
else:
    %cd /content/cerebro_spheroid_v5
    !git pull origin main

%cd /content/cerebro_spheroid_v5

# Install minimal fast training prerequisites
!pip install -q transformers datasets accelerate sentencepiece pyarrow

## 3. Select Model Architecture Tier
Choose which CerebroSpheroid model tier to train or resume:
- **`v5_small` (101.6M params, 16 shells)**: Native 16-shell geometry, directly compatible with salvaged V4 checkpoints.
- **`v5_compact` (244.4M params, 32 shells)**: High-capacity multi-shell cognitive model.
- **`v5_base` (617.3M params, 48 shells)**: Frontier-scale capacity.

In [ ]:
#@title Model Architecture Tier Selection
MODEL_PRESET = "v5_small" #@param ["v5_small", "v5_compact", "v5_base"]
BATCH_SIZE = 4 #@param {type:"integer"}
GRAD_ACCUM = 8 #@param {type:"integer"}
TOTAL_WINDOW = 1024 #@param {type:"integer"}

print(f"✓ Selected Model Tier: {MODEL_PRESET}")
print(f"✓ Effective Batch:   {BATCH_SIZE * GRAD_ACCUM} sequences ({BATCH_SIZE * GRAD_ACCUM * (TOTAL_WINDOW // 2):,} tokens/step)")

## 4. Multi-Model Cloud Hub Status & Checkpoint Resumption
Queries `https://bioflo.in/api/cerebro/sync.php` for available checkpoints for this model tier (`latest` and `best`).

In [ ]:
from src.cloud_sync import check_remote_manifest, pull_checkpoint

manifest = check_remote_manifest(model=MODEL_PRESET)
print(f"Sync Hub Online: {manifest.get(status) == online}")
models_data = manifest.get(models, {})
print("\n--- Centralized Model Registry Status ---")
for m_name in ["v5_small", "v5_compact", "v5_base"]:
    info = models_data.get(m_name, {})
    active_mark = " (ACTIVE)" if m_name == MODEL_PRESET else ""
    print(f"  • {m_name:10s}{active_mark}: Latest: {info.get(has_latest, False)} ({info.get(latest_size, 0)/1e6:.1f} MB) | Best: {info.get(has_best, False)} ({info.get(best_size, 0)/1e6:.1f} MB)")

# Prepare local checkpoint directory
ckpt_dir = f"/content/cerebro_spheroid_v5/checkpoints/{MODEL_PRESET}"
os.makedirs(ckpt_dir, exist_ok=True)
local_latest = f"{ckpt_dir}/{MODEL_PRESET}_latest.pt"

# Auto-pull latest if available remotely
active_info = models_data.get(MODEL_PRESET, {})
if active_info.get("has_latest", False):
    print(f"\nRestoring latest {MODEL_PRESET} checkpoint from cloud hub...")
    pull_checkpoint(local_latest, target_type="latest", model=MODEL_PRESET)
else:
    print(f"\nNo remote checkpoint found for {MODEL_PRESET}. Will start fresh training or use local weights.")

## 5. Training by Dataset Mixture
Select and run one of the cells below depending on your training objective.

### 🔄 Option 5A: Full Graded Developmental Curriculum (`--mix auto`)
Trains across the complete 4-phase Warmup-Stable-Decay (WSD) developmental schedule.

In [ ]:
# Option 5A: Full Graded Developmental Curriculum
!python3 scripts/train_v5_graded_curriculum.py \
    --model-preset $MODEL_PRESET \
    --mix auto \
    --batch-size $BATCH_SIZE \
    --grad-accum $GRAD_ACCUM \
    --elastic-window \
    --total-window $TOTAL_WINDOW \
    --max-lr 3.5e-4 \
    --save-interval 250 \
    --cloud-sync \
    --cloud-sync-interval 100

### 📚 Option 5B: Mixture 1 - Grammar, Lexical Syntax & Educational Prose (`--mix foundation`)
High-density Cosmopedia v2 + FineWeb-Edu + Clean Wikipedia. Grounds vocabulary and syntax.

In [ ]:
# Option 5B: Foundation & Grammar Prose
!python3 scripts/train_v5_graded_curriculum.py \
    --model-preset $MODEL_PRESET \
    --mix foundation \
    --batch-size $BATCH_SIZE \
    --grad-accum $GRAD_ACCUM \
    --elastic-window \
    --total-window $TOTAL_WINDOW \
    --max-lr 3.5e-4 \
    --save-interval 250 \
    --cloud-sync \
    --cloud-sync-interval 100

### 🛠️ Option 5C: Mixture 2 - Multi-Step Mathematics & Deductive Reasoning (`--mix reasoning`)
GSM8K + Open-Web-Math + OpenHermes-2.5 multi-step Chain-of-Thought.

In [ ]:
# Option 5C: Mathematics & Deductive Reasoning
!python3 scripts/train_v5_graded_curriculum.py \
    --model-preset $MODEL_PRESET \
    --mix reasoning \
    --batch-size $BATCH_SIZE \
    --grad-accum $GRAD_ACCUM \
    --elastic-window \
    --total-window $TOTAL_WINDOW \
    --max-lr 3.0e-4 \
    --save-interval 250 \
    --cloud-sync \
    --cloud-sync-interval 100

### 💻 Option 5D: Mixture 3 - Python Code, Algorithms & Unit Tests (`--mix code`)
The Stack Python-Edu + Algorithmic derivations.

In [ ]:
# Option 5D: Python Code & Algorithms
!python3 scripts/train_v5_graded_curriculum.py \
    --model-preset $MODEL_PRESET \
    --mix code \
    --batch-size $BATCH_SIZE \
    --grad-accum $GRAD_ACCUM \
    --elastic-window \
    --total-window $TOTAL_WINDOW \
    --max-lr 3.0e-4 \
    --save-interval 250 \
    --cloud-sync \
    --cloud-sync-interval 100

### 🤖 Option 5E: Mixture 4 - Modern Agentic Workflows & Multi-Turn Tool Execution (`--mix agentic`)
SmolTalk + Hermes Function Calling (`BASH`, `PYTHON`, `CALCULATOR`, `FILE_READ`) + UltraChat multi-turn.

In [ ]:
# Option 5E: Agentic Workflows & Tool Execution
!python3 scripts/train_v5_graded_curriculum.py \
    --model-preset $MODEL_PRESET \
    --mix agentic \
    --batch-size $BATCH_SIZE \
    --grad-accum $GRAD_ACCUM \
    --elastic-window \
    --total-window $TOTAL_WINDOW \
    --max-lr 2.5e-4 \
    --save-interval 250 \
    --cloud-sync \
    --cloud-sync-interval 100

### ⭐ Option 5F: Mixture 5 - Golden Consolidation & Anti-Hallucination Replay (`--mix consolidation`)
Balanced 4-pillar replay + CerebroCognitive-10k (strawberry letter counting, temporal clock reasoning, counterfactual anti-sycophancy).

In [ ]:
# Option 5F: Golden Consolidation Replay
!python3 scripts/train_v5_graded_curriculum.py \
    --model-preset $MODEL_PRESET \
    --mix consolidation \
    --batch-size $BATCH_SIZE \
    --grad-accum $GRAD_ACCUM \
    --elastic-window \
    --total-window $TOTAL_WINDOW \
    --max-lr 2.0e-4 \
    --save-interval 250 \
    --cloud-sync \
    --cloud-sync-interval 100

## 6. Evaluate on Standardized Cerebro-1000 Benchmark
Evaluates the active model tier against the 1,000 fixed benchmark prompts across Language, Knowledge, Math, Code, and Tools.

In [ ]:
# Run benchmark pulse check on the active model
active_ckpt = f"checkpoints/{MODEL_PRESET}/{MODEL_PRESET}_latest.pt"
if not os.path.exists(active_ckpt):
    active_ckpt = f"checkpoints/{MODEL_PRESET}/latest.pt"

!python3 eval/run_benchmark.py \
    --checkpoint $active_ckpt \
    --samples 100

# Display Markdown Benchmark Report
from IPython.display import display, Markdown
if os.path.exists("eval/results/BENCHMARK_REPORT.md"):
    with open("eval/results/BENCHMARK_REPORT.md") as f:
        display(Markdown(f.read()))